# Graph Explorer

This notebook loads `graph.ttl` from the project root and lets you explore the knowledge graph with SPARQL.

If the graph has not been built yet, run:

```bash
python3 pipeline/run.py --to build
```

The helper cells below give you:

- a reusable `run_query()` function for arbitrary SPARQL
- a `find_exercise()` helper for looking up exercise labels and URIs
- a `describe_exercise()` helper for browsing all triples about one exercise by label
- sample competency-question queries you can adapt


In [1]:
from pathlib import Path
import html

import pyoxigraph as ox
from IPython.display import HTML, Markdown, display

PROJECT_ROOT = Path.cwd()
GRAPH_PATH = PROJECT_ROOT / "graph.ttl"
FEG = "https://placeholder.url#"

if not GRAPH_PATH.exists():
    raise FileNotFoundError(
        f"{GRAPH_PATH} does not exist. Run `python3 pipeline/run.py --to build` first."
    )

store = ox.Store()
store.load(GRAPH_PATH.read_bytes(), format=ox.RdfFormat.TURTLE)

triple_count = next(iter(store.query("SELECT (COUNT(*) AS ?n) WHERE { ?s ?p ?o }")))["n"].value
print(f"Loaded {GRAPH_PATH.name} with {triple_count} triples.")


Loaded graph.ttl with 242642 triples.


In [2]:
PREFIXES = f"""
PREFIX feg:  <{FEG}>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
""".strip()


def term_value(term):
    if term is None:
        return None
    return getattr(term, "value", str(term))


def render_rows(rows, columns, max_rows=25):
    if not rows:
        display(Markdown("_No rows returned._"))
        return

    visible = rows[:max_rows]
    head = "".join(f"<th>{html.escape(col)}</th>" for col in columns)
    body_rows = []
    for row in visible:
        body_rows.append(
            "<tr>" + "".join(
                f"<td>{html.escape('' if row.get(col) is None else str(row.get(col)))}</td>"
                for col in columns
            ) + "</tr>"
        )

    table = (
        "<table>"
        f"<thead><tr>{head}</tr></thead>"
        f"<tbody>{''.join(body_rows)}</tbody>"
        "</table>"
    )
    display(HTML(table))

    if len(rows) > max_rows:
        display(Markdown(f"_Showing first {max_rows} of {len(rows)} rows._"))


def run_query(query: str, max_rows: int = 25):
    result = store.query(query)
    columns = [str(var).lstrip("?") for var in result.variables]
    rows = []
    for solution in result:
        rows.append({col: term_value(solution[col]) for col in columns})

    print(f"{len(rows)} row(s)")
    render_rows(rows, columns, max_rows=max_rows)
    return rows


def find_exercise(name_fragment: str, limit: int = 20):
    escaped = name_fragment.lower().replace('"', '\\"')
    query = PREFIXES + f"""
SELECT ?exercise ?name
WHERE {{
    ?exercise a feg:Exercise ;
              rdfs:label ?name .
    FILTER(CONTAINS(LCASE(STR(?name)), \"{escaped}\"))
}}
ORDER BY ?name
LIMIT {limit}
"""
    return run_query(query, max_rows=limit)


def _resolve_exercise_uri_by_label(exercise_label: str):
    escaped = exercise_label.replace('"', '\\"')
    query = PREFIXES + f"""
SELECT ?exercise ?name
WHERE {{
    ?exercise a feg:Exercise ;
              rdfs:label ?name .
    FILTER(LCASE(STR(?name)) = LCASE(\"{escaped}\"))
}}
ORDER BY ?exercise
"""
    rows = run_query(query, max_rows=25)
    if not rows:
        raise ValueError(
            f"No exercise found with exact label {exercise_label!r}. Try find_exercise() first."
        )
    if len(rows) > 1:
        raise ValueError(
            f"Label {exercise_label!r} is ambiguous. Use find_exercise() to choose one exact label."
        )
    return rows[0]["exercise"]


def describe_exercise(exercise_label: str, max_rows: int = 50):
    exercise_uri = _resolve_exercise_uri_by_label(exercise_label)
    query = PREFIXES + f"""
SELECT ?predicate ?object
WHERE {{
    <{exercise_uri}> ?predicate ?object .
}}
ORDER BY ?predicate ?object
"""
    return run_query(query, max_rows=max_rows)


## Ad hoc SPARQL

Edit the query string below and rerun the cell whenever you want to inspect the graph directly.


In [3]:
QUERY = PREFIXES + """
SELECT ?exercise ?name
WHERE {
    ?exercise a feg:Exercise ;
              rdfs:label ?name .
}
ORDER BY ?name
LIMIT 10
"""

run_query(QUERY)


10 row(s)


exercise,name
https://placeholder.url#ex_fed_3_4_Sit_Up,3/4 Sit-Up
https://placeholder.url#ex_fed_90_90_Hamstring,90/90 Hamstring
https://placeholder.url#ex_fed_Ab_Crunch_Machine,Ab Crunch Machine
https://placeholder.url#ex_fed_Ab_Roller,Ab Roller
https://placeholder.url#ex_ffdb_Ab_Wheel_Kneeling_Rollout,Ab Wheel Kneeling Rollout
https://placeholder.url#ex_ffdb_Ab_Wheel_Standing_Rollout,Ab Wheel Standing Rollout
https://placeholder.url#ex_fed_Adductor,Adductor
https://placeholder.url#ex_fed_Adductor_Groin,Adductor/Groin
https://placeholder.url#ex_fed_Advanced_Kettlebell_Windmill,Advanced Kettlebell Windmill
https://placeholder.url#ex_fed_Air_Bike,Air Bike


[{'exercise': 'https://placeholder.url#ex_fed_3_4_Sit_Up',
  'name': '3/4 Sit-Up'},
 {'exercise': 'https://placeholder.url#ex_fed_90_90_Hamstring',
  'name': '90/90 Hamstring'},
 {'exercise': 'https://placeholder.url#ex_fed_Ab_Crunch_Machine',
  'name': 'Ab Crunch Machine'},
 {'exercise': 'https://placeholder.url#ex_fed_Ab_Roller', 'name': 'Ab Roller'},
 {'exercise': 'https://placeholder.url#ex_ffdb_Ab_Wheel_Kneeling_Rollout',
  'name': 'Ab Wheel Kneeling Rollout'},
 {'exercise': 'https://placeholder.url#ex_ffdb_Ab_Wheel_Standing_Rollout',
  'name': 'Ab Wheel Standing Rollout'},
 {'exercise': 'https://placeholder.url#ex_fed_Adductor', 'name': 'Adductor'},
 {'exercise': 'https://placeholder.url#ex_fed_Adductor_Groin',
  'name': 'Adductor/Groin'},
 {'exercise': 'https://placeholder.url#ex_fed_Advanced_Kettlebell_Windmill',
  'name': 'Advanced Kettlebell Windmill'},
 {'exercise': 'https://placeholder.url#ex_fed_Air_Bike', 'name': 'Air Bike'}]

## Competency Questions

These starter queries answer basic competency questions about the graph and double as templates you can adapt.


### 1. How many exercises are in the graph?


In [4]:
exercise_count_query = PREFIXES + """
SELECT (COUNT(DISTINCT ?exercise) AS ?exerciseCount)
WHERE {
    ?exercise a feg:Exercise .
}
"""

run_query(exercise_count_query)


1 row(s)


exerciseCount
3442


[{'exerciseCount': '3442'}]

### 2. Which movement patterns are most represented?


In [5]:
top_patterns_query = PREFIXES + """
SELECT ?patternLabel (COUNT(DISTINCT ?exercise) AS ?exerciseCount)
WHERE {
    ?exercise a feg:Exercise ;
              feg:movementPattern ?pattern .
    ?pattern rdfs:label ?patternLabel .
}
GROUP BY ?patternLabel
ORDER BY DESC(?exerciseCount)
LIMIT 15
"""

run_query(top_patterns_query)


15 row(s)


patternLabel,exerciseCount
Knee Dominant,1112
Hip Hinge,416
Vertical Push,383
Rotation,252
Horizontal Push,238
Isometric Hold,175
Anti Extension,134
Horizontal Pull,131
Mobility,112
Vertical Pull,91


[{'patternLabel': 'Knee Dominant', 'exerciseCount': '1112'},
 {'patternLabel': 'Hip Hinge', 'exerciseCount': '416'},
 {'patternLabel': 'Vertical Push', 'exerciseCount': '383'},
 {'patternLabel': 'Rotation', 'exerciseCount': '252'},
 {'patternLabel': 'Horizontal Push', 'exerciseCount': '238'},
 {'patternLabel': 'Isometric Hold', 'exerciseCount': '175'},
 {'patternLabel': 'Anti Extension', 'exerciseCount': '134'},
 {'patternLabel': 'Horizontal Pull', 'exerciseCount': '131'},
 {'patternLabel': 'Mobility', 'exerciseCount': '112'},
 {'patternLabel': 'Vertical Pull', 'exerciseCount': '91'},
 {'patternLabel': 'Locomotion', 'exerciseCount': '89'},
 {'patternLabel': 'Anti Lateral Flexion', 'exerciseCount': '86'},
 {'patternLabel': 'Squat', 'exerciseCount': '62'},
 {'patternLabel': 'Core Flexion', 'exerciseCount': '46'},
 {'patternLabel': 'Anti Rotation', 'exerciseCount': '37'}]

### 3. Which exercises target the shoulder region as a prime mover?

This query walks down the SKOS muscle hierarchy from `feg:Shoulders`, so it will match region-, group-, and head-level muscle terms.


In [6]:
shoulders_query = PREFIXES + """
SELECT ?name ?muscleLabel
WHERE {
    BIND(feg:Shoulders AS ?targetRegion)

    ?exercise a feg:Exercise ;
              rdfs:label ?name ;
              feg:hasInvolvement ?inv .

    ?inv feg:degree feg:PrimeMover ;
         feg:muscle ?muscle .

    ?muscle skos:broader* ?targetRegion .
    OPTIONAL { ?muscle rdfs:label ?muscleLabel }
}
ORDER BY ?name ?muscleLabel
LIMIT 25
"""

run_query(shoulders_query)


25 row(s)


name,muscleLabel
Alternating Cable Shoulder Press,Anterior Deltoid
Alternating Cable Shoulder Press,Lateral Deltoid
Alternating Deltoid Raise,Anterior Deltoid
Alternating Deltoid Raise,Lateral Deltoid
Alternating Double Clubbell Front Flag Press,Anterior Deltoid
Alternating Double Clubbell Shield Cast,Anterior Deltoid
Alternating Double Clubbell Shield Cast,Posterior Deltoid
Alternating Double Clubbell Side Flag Press,Lateral Deltoid
Alternating Double Clubbell Torch Press,Anterior Deltoid
Alternating Double Dumbbell Arnold Press,Anterior Deltoid


[{'name': 'Alternating Cable Shoulder Press',
  'muscleLabel': 'Anterior Deltoid'},
 {'name': 'Alternating Cable Shoulder Press',
  'muscleLabel': 'Lateral Deltoid'},
 {'name': 'Alternating Deltoid Raise', 'muscleLabel': 'Anterior Deltoid'},
 {'name': 'Alternating Deltoid Raise', 'muscleLabel': 'Lateral Deltoid'},
 {'name': 'Alternating Double Clubbell Front Flag Press',
  'muscleLabel': 'Anterior Deltoid'},
 {'name': 'Alternating Double Clubbell Shield Cast',
  'muscleLabel': 'Anterior Deltoid'},
 {'name': 'Alternating Double Clubbell Shield Cast',
  'muscleLabel': 'Posterior Deltoid'},
 {'name': 'Alternating Double Clubbell Side Flag Press',
  'muscleLabel': 'Lateral Deltoid'},
 {'name': 'Alternating Double Clubbell Torch Press',
  'muscleLabel': 'Anterior Deltoid'},
 {'name': 'Alternating Double Dumbbell Arnold Press',
  'muscleLabel': 'Anterior Deltoid'},
 {'name': 'Alternating Double Dumbbell Arnold Press',
  'muscleLabel': 'Lateral Deltoid'},
 {'name': 'Alternating Double Dumbbel

### 4. Which bodyweight anti-rotation exercises are in the graph?

This is a simple example of combining a movement pattern with an equipment filter.


In [7]:
bodyweight_antirotation_query = PREFIXES + """
SELECT ?name
WHERE {
    ?exercise a feg:Exercise ;
              rdfs:label ?name ;
              feg:movementPattern feg:AntiRotation ;
              feg:equipment feg:Bodyweight .
}
ORDER BY ?name
LIMIT 25
"""

run_query(bodyweight_antirotation_query)


5 row(s)


name
Bodyweight Bird Dog
Bodyweight Bird Dog Knee to Elbow
Bodyweight Ipsilateral Bird Dog
Bodyweight Knee Hover Bird Dog
Single-Arm Push-Up


[{'name': 'Bodyweight Bird Dog'},
 {'name': 'Bodyweight Bird Dog Knee to Elbow'},
 {'name': 'Bodyweight Ipsilateral Bird Dog'},
 {'name': 'Bodyweight Knee Hover Bird Dog'},
 {'name': 'Single-Arm Push-Up'}]

### 5. What are plausible substitutions for Dead Bug?

This example uses the current `Dead Bug` node directly in SPARQL, then the label-based helper below shows the human-friendly browse path.


In [ ]:
find_exercise("dead bug")


In [8]:
dead_bug_substitutions_query = PREFIXES + """
SELECT ?candidate ?name (COUNT(DISTINCT ?sharedMuscle) AS ?sharedPrimeMovers)
WHERE {
    BIND(feg:ex_dead_bug AS ?source)

    ?source feg:movementPattern ?pattern ;
            feg:hasInvolvement ?srcInv .
    ?srcInv feg:degree feg:PrimeMover ;
            feg:muscle ?sharedMuscle .

    ?candidate feg:movementPattern ?pattern ;
               rdfs:label ?name ;
               feg:hasInvolvement ?candInv .
    ?candInv feg:degree feg:PrimeMover ;
             feg:muscle ?sharedMuscle .

    FILTER(?candidate != ?source)
}
GROUP BY ?candidate ?name
ORDER BY DESC(?sharedPrimeMovers) ?name
LIMIT 15
"""

run_query(dead_bug_substitutions_query)


15 row(s)


candidate,name,sharedPrimeMovers
https://placeholder.url#ex_fed_Ab_Roller,Ab Roller,2
https://placeholder.url#ex_ffdb_Ab_Wheel_Kneeling_Rollout,Ab Wheel Kneeling Rollout,2
https://placeholder.url#ex_ffdb_Ab_Wheel_Standing_Rollout,Ab Wheel Standing Rollout,2
https://placeholder.url#ex_alternating_single_arm_feet_elevated_plank_pull_through,Alternating Single Arm Dumbbell Feet Elevated Plank Pull Through,2
https://placeholder.url#ex_alternating_single_arm_plank_pull_through,Alternating Single Arm Dumbbell Plank Pull Through,2
https://placeholder.url#ex_ffdb_Alternating_Single_Arm_Kettlebell_Knee_Hover_Quadruped_Pull_Through,Alternating Single Arm Kettlebell Knee Hover Quadruped Pull Through,2
https://placeholder.url#ex_ffdb_Bar_Hanging_Toes_to_Bar,Bar Hanging Toes to Bar,2
https://placeholder.url#ex_fed_Barbell_Ab_Rollout,Barbell Ab Rollout,2
https://placeholder.url#ex_fed_Barbell_Ab_Rollout___On_Knees,Barbell Ab Rollout - On Knees,2
https://placeholder.url#ex_kneeling_rollout,Barbell Kneeling Rollout,2


[{'candidate': 'https://placeholder.url#ex_fed_Ab_Roller',
  'name': 'Ab Roller',
  'sharedPrimeMovers': '2'},
 {'candidate': 'https://placeholder.url#ex_ffdb_Ab_Wheel_Kneeling_Rollout',
  'name': 'Ab Wheel Kneeling Rollout',
  'sharedPrimeMovers': '2'},
 {'candidate': 'https://placeholder.url#ex_ffdb_Ab_Wheel_Standing_Rollout',
  'name': 'Ab Wheel Standing Rollout',
  'sharedPrimeMovers': '2'},
 {'candidate': 'https://placeholder.url#ex_alternating_single_arm_feet_elevated_plank_pull_through',
  'name': 'Alternating Single Arm Dumbbell Feet Elevated Plank Pull Through',
  'sharedPrimeMovers': '2'},
 {'candidate': 'https://placeholder.url#ex_alternating_single_arm_plank_pull_through',
  'name': 'Alternating Single Arm Dumbbell Plank Pull Through',
  'sharedPrimeMovers': '2'},
 {'candidate': 'https://placeholder.url#ex_ffdb_Alternating_Single_Arm_Kettlebell_Knee_Hover_Quadruped_Pull_Through',
  'name': 'Alternating Single Arm Kettlebell Knee Hover Quadruped Pull Through',
  'sharedPrime

### 6. What does the graph know about one exercise?

Use the exercise label, not the URI.


In [9]:
describe_exercise("Dead Bug")


1 row(s)


exercise,name
https://placeholder.url#ex_dead_bug,Dead Bug


25 row(s)


predicate,object
http://purl.org/dc/terms/source,https://placeholder.url#FreeExerciseDB
http://purl.org/dc/terms/source,https://placeholder.url#FunctionalFitnessDB
http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://placeholder.url#Exercise
http://www.w3.org/2000/01/rdf-schema#label,Dead Bug
https://placeholder.url#equipment,https://placeholder.url#Bodyweight
https://placeholder.url#equipment,https://placeholder.url#Kettlebell
https://placeholder.url#exerciseStyle,https://placeholder.url#Postural
https://placeholder.url#hasInvolvement,https://placeholder.url#inv_ex_dead_bug_ErectorSpinae_Stabilizer
https://placeholder.url#hasInvolvement,https://placeholder.url#inv_ex_dead_bug_GluteusMaximus_Synergist
https://placeholder.url#hasInvolvement,https://placeholder.url#inv_ex_dead_bug_Iliopsoas_Stabilizer


[{'predicate': 'http://purl.org/dc/terms/source',
  'object': 'https://placeholder.url#FreeExerciseDB'},
 {'predicate': 'http://purl.org/dc/terms/source',
  'object': 'https://placeholder.url#FunctionalFitnessDB'},
 {'predicate': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type',
  'object': 'https://placeholder.url#Exercise'},
 {'predicate': 'http://www.w3.org/2000/01/rdf-schema#label',
  'object': 'Dead Bug'},
 {'predicate': 'https://placeholder.url#equipment',
  'object': 'https://placeholder.url#Bodyweight'},
 {'predicate': 'https://placeholder.url#equipment',
  'object': 'https://placeholder.url#Kettlebell'},
 {'predicate': 'https://placeholder.url#exerciseStyle',
  'object': 'https://placeholder.url#Postural'},
 {'predicate': 'https://placeholder.url#hasInvolvement',
  'object': 'https://placeholder.url#inv_ex_dead_bug_ErectorSpinae_Stabilizer'},
 {'predicate': 'https://placeholder.url#hasInvolvement',
  'object': 'https://placeholder.url#inv_ex_dead_bug_GluteusMaximus_Synergist'